01 - data generation and normalized event schema

In [1]:
import json
import random
import uuid
from dataclasses import dataclass, asdict, field
from datetime import datetime, timedelta
from typing import Optional, List, Literal

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid", palette="mako")
plt.rcParams["figure.figsize"] = (11, 4)

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

print("Libraries loaded. Seed fixed at", RNG_SEED)

Libraries loaded. Seed fixed at 42


1. normalized event schema

In [3]:
EVENT_TYPES = Literal["authentication","network_connection","process_execution","logon","logoff","dns_query","file_access"]

@dataclass
class EventSchema:
    '''
    Normalized internal event schema. Dataset-agnostic.

    Any raw log source (LANL, DARPA OpTC, synthetic, Zeek/Sysmon, etc.)
    must be mapped into this shape by a small per-dataset adapter function
    before entering the graph builder
    '''

    event_id: str
    timestamp: str
    source: str
    destination: str
    user: Optional[str]
    event_type: str
    protocol: Optional[str] = None
    port: Optional[int] = None
    auth_type: Optional[str] = None
    duration_sec: Optional[float] = None
    success: Optional[bool] = True
    severity: Optional[str] = "info"
    is_malicious: bool = False
    technique_id: Optional[str] = None
    attack_stage: Optional[int] = None

    def to_dict(self):
        return asdict(self)

# test
_sample = EventSchema(
    event_id=str(uuid.uuid4()), timestamp=datetime.utcnow().isoformat(),
    source="APP-01", destination="DC-01", user="svc_backup",
    event_type="authentication", protocol="SMB", port=445, auth_type="NTLM",
    success=True,
)
print(json.dumps(_sample.to_dict(), indent=2))

{
  "event_id": "d184fb90-bf7a-44d5-b3e3-f4b85a1a2098",
  "timestamp": "2026-09-09T15:29:02.644372",
  "source": "APP-01",
  "destination": "DC-01",
  "user": "svc_backup",
  "event_type": "authentication",
  "protocol": "SMB",
  "port": 445,
  "auth_type": "NTLM",
  "duration_sec": null,
  "success": true,
  "severity": "info",
  "is_malicious": false,
  "technique_id": null,
  "attack_stage": null
}
